# Exercises XP: Day 3 - BERT in Practice

## Exercise 1 - Tokenization with BERT

In [10]:
# Optional setup: install dependencies if they are missing in your environment.
%pip install -q transformers torch


Note: you may need to restart the kernel to use updated packages.


In [11]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sample_sentence = "AI models retrieve documents before generating answers"
print(sample_sentence)


AI models retrieve documents before generating answers


In [12]:
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=18,  # TODO: adjust if your sentence needs more room
    return_attention_mask=True,
    return_tensors="pt"
)

input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print("index | token        | id")
print("-------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    print(f"{idx:>5} | {token:<12} | {token_id:>5}")

print("\nAttention mask:", encoding["attention_mask"][0].tolist())
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):", special_positions)


index | token        | id
-------------------------
    0 | [CLS]        |   101
    1 | ai           |  9932
    2 | models       |  4275
    3 | retrieve     | 12850
    4 | documents    |  5491
    5 | before       |  2077
    6 | generating   | 11717
    7 | answers      |  6998
    8 | [SEP]        |   102
    9 | [PAD]        |     0
   10 | [PAD]        |     0
   11 | [PAD]        |     0
   12 | [PAD]        |     0
   13 | [PAD]        |     0
   14 | [PAD]        |     0
   15 | [PAD]        |     0
   16 | [PAD]        |     0
   17 | [PAD]        |     0

Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Special tokens (index, token): [(0, '[CLS]'), (8, '[SEP]'), (9, '[PAD]'), (10, '[PAD]'), (11, '[PAD]'), (12, '[PAD]'), (13, '[PAD]'), (14, '[PAD]'), (15, '[PAD]'), (16, '[PAD]'), (17, '[PAD]')]


### Exercise 1 reflection
- Describe how [CLS] and [SEP] behave inside the encoder.

  *[CLS] token: It is added at the beginning of the input. Its final embedding is used as a summary of the whole sequence.

  *[SEP] token: It is used to separate sentences or mark the end of a sequence. It helps the encoder understand sentence boundaries.

- Explain how the attention mask hides padded positions from self-attention.\n

  *Attention mask: The attention mask tells the model which tokens are real and which are padding. Padded positions are masked out so the model does not attend to them during self-attention.

## Exercise 2 - Sentiment analysis pipeline

In [14]:
from transformers import pipeline


sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sentence = "I really enjoyed this movie, it was amazing!"
prediction = sentiment_pipeline(sentence)
print(prediction)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:  37%|###6      | 157M/425M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.9998784065246582}]


### Exercise 2 reflection
- Does the predicted label match your expectation? Why or why not?

  yes, the predicted label match my expectation because the sentence I gave is representing a positive impression about the movie.

- How confident is the model and what does the score tell you?

  the model tells me that it is 99% confident and that's good so far.

## Exercise 3 - Custom sentiment analyzer class

In [19]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict

class BERTSentimentAnalyzer:
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
        # device (GPU if available, else CPU)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.max_length = max_length

        # load tokenizer and model
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        # basic cleaning
        text = text.strip()

        # tokenize and convert to tensors
        inputs = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        # move tensors to device
        return {k: v.to(self.device) for k, v in inputs.items()}

    def predict(self, text: str) -> Dict[str, float]:
        inputs = self.preprocess(text)

        with torch.no_grad():
            outputs = self.model(**inputs)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=-1)

        # get prediction
        score, label_id = torch.max(probs, dim=-1)
        label = self.model.config.id2label[label_id.item()]

        return {
            "label": label,
            "score": score.item()
        }


In [20]:
# instantiate the analyzer
analyzer = BERTSentimentAnalyzer()

# sample sentences
samples = [
    "This product is excellent and works perfectly.",
    "I am very disappointed, the experience was terrible."
]

# test predictions
for text in samples:
    print(text)
    print(analyzer.predict(text))
    print()



This product is excellent and works perfectly.
{'label': 'POSITIVE', 'score': 0.9998756647109985}

I am very disappointed, the experience was terrible.
{'label': 'NEGATIVE', 'score': 0.9997096657752991}



## Exercise 4 - BERT for Named Entity Recognition

In [21]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        # detect device
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # load tokenizer and model
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()

        # label mapping
        self.id2label = self.model.config.id2label

    def recognize(self, text: str):
        # tokenize input
        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            return_offsets_mapping=True
        )

        offset_mapping = inputs.pop("offset_mapping")[0]
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        # forward pass
        with torch.no_grad():
            outputs = self.model(**inputs)

        predictions = torch.argmax(outputs.logits, dim=-1)[0].cpu().numpy()
        tokens = self.tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

        entities = []
        current_entity = None

        for token, label_id, offsets in zip(tokens, predictions, offset_mapping):
            label = self.id2label[label_id]
            start, end = offsets.tolist()

            # skip special tokens
            if start == end:
                continue

            if label.startswith("B-"):
                if current_entity:
                    entities.append(current_entity)

                current_entity = {
                    "text": text[start:end],
                    "entity": label[2:],
                    "start": start,
                    "end": end
                }

            elif label.startswith("I-") and current_entity:
                current_entity["text"] += text[start:end]
                current_entity["end"] = end

            else:
                if current_entity:
                    entities.append(current_entity)
                    current_entity = None

        if current_entity:
            entities.append(current_entity)

        return entities


In [22]:
# instantiate the recognizer
ner = BERTNamedEntityRecognizer()

# sample text with multiple entity types
sample_text = (
    "Elon Musk founded SpaceX in California before becoming the CEO of Tesla."
)

# run NER
entities = ner.recognize(sample_text)

for entity in entities:
    print(entity)

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


{'text': 'El', 'entity': 'PER', 'start': 0, 'end': 2}
{'text': 'onMusk', 'entity': 'PER', 'start': 2, 'end': 9}
{'text': 'SpaceX', 'entity': 'ORG', 'start': 18, 'end': 24}
{'text': 'California', 'entity': 'LOC', 'start': 28, 'end': 38}
{'text': 'Tesla', 'entity': 'ORG', 'start': 66, 'end': 71}


## Exercise 5 - Comparing BERT and GPT
| Category | BERT | GPT |
|----------|------|-----|
| Architecture | Bidirectional Transformer (Encoder) | Autoregressive Transformer (Decoder) |
| Primary purpose | Understanding language | Generating language |
| Typical use cases | Sentiment analysis, named entity recognition, search | Chatbots, creative writing, code generation |
| Strengths | Excels at understanding context from both sides of a word | Excellent at fluent, coherent text generation |
| Weaknesses | Poor at text generation tasks | Can "hallucinate" facts; not designed for classification |

## Exercise 6 - BERT inside Retrieval-Augmented Generation
### How BERT-Generated Embeddings Power the Retrieval Stage of a RAG Workflow

##### 1. Encoding Text into Meaning
BERT processes text by reading entire sentences bidirectionally. It generates a unique numerical fingerprint, called an embedding, that captures the semantic meaning of a query or a document passage.

##### 2. Storing and Matching Concepts
These meaning-rich embeddings are stored in a specialized vector database. When a query is received, the database instantly performs a similarity search to find the stored document vectors that most closely align with the query's conceptual fingerprint.

##### 3. Providing Grounded Context to the Generator
The top-ranked text passages retrieved by this search are passed, along with the original query, to a generative model like GPT. This provides the generator with factual, relevant source material to base its response on.

##### 4. Concrete Application: Enterprise Knowledge Assistant
In a corporate setting, an employee might ask, "What's the process for a hardware upgrade request?" The RAG system uses BERT to instantly retrieve the official procurement policy and relevant forms. GPT then synthesizes this information into a clear, actionable summary for the employee.
